This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data.

In [ ]:
import os
import sys
from pathlib import Path
# Data processing and analysis
import pandas as pd
import numpy as np

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring/scoring_system.py
    
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git

In [8]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

import corrosion_scoring as cs

In [3]:
# large galaxies input and output #large size dir for large files hosted instead in kaggle
large_dir = Path("/home/beatriz/MIC")
# Directory to output large files # eccontris, compilated dbs
output_large = large_dir / "output_large"
# Whole filtered Data
eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'

In [4]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

In [5]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = [
            'idx', 'Genus', 'protein_name', 'EC', 'enzyme_names',
            'enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'corrosion_relevance', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]
    
    for d in global_terms_list:
        # If the dict is nested, flatten it or handle as needed
        for category, terms in d.items():
            # If terms is itself a dict, flatten or handle recursively
            if isinstance(terms, dict):
                for subcategory, subterms in terms.items():
                    existing = []
                    for term in subterms:
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[f"{category}.{subcategory}"] = existing
            else:
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    return found


In [ ]:
found = validate_terms(ECcontri_Uniprot_enriched, [
    cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups,
    cs.metal_mapping,
])